# Import and Read data


In [1]:
import sys
sys.path.append('.')  # Add current directory to path
from haversine_build_graph_and_train import *
import torch

/opt/conda/lib/python3.11/site-packages/transformers/utils/hub.py:127: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
#partition = 100

In [3]:
#df = pd.read_csv(f"../../../data/top30groups/anonLoc/combined/combined{partition}.csv")

In [4]:
"""from sklearn.preprocessing import StandardScaler

# Columns to exclude from scaling
exclude_cols = ['gname']

# Columns to scale
scale_cols = [col for col in df.columns if col not in exclude_cols]

# Scale only selected columns
scaler = StandardScaler()
df[scale_cols] = scaler.fit_transform(df[scale_cols])"""

"from sklearn.preprocessing import StandardScaler\n\n# Columns to exclude from scaling\nexclude_cols = ['gname']\n\n# Columns to scale\nscale_cols = [col for col in df.columns if col not in exclude_cols]\n\n# Scale only selected columns\nscaler = StandardScaler()\ndf[scale_cols] = scaler.fit_transform(df[scale_cols])"

# Weapon type prediction

In [5]:
torch.cuda.empty_cache()

In [6]:
y_preds, y_trues, logs = [], [], []


import itertools, random, os

param_grid = {
    'lr': [0.001, 0.01],
    'n_tree': [40, 80, 100],
    'tree_depth': [8, 9, 10, 11, 12],
    'tree_feature_rate': [0.1, 0.3, 0.5],
    'feat_dropout': [0.0, 0.1, 0.2],
    'embed_dim': [32, 64]
}

# Random sample from grid
keys = list(param_grid.keys())
all_combos = list(itertools.product(*param_grid.values()))
sample_size = min(10, len(all_combos))


partitions = [300]
best_scores = []
best_runs = []
best_paramsets = []

for partition in partitions:

    sampled_combos = random.sample(all_combos, sample_size)
    sampled_dicts = [dict(zip(keys, combo)) for combo in sampled_combos]

    df = pd.read_csv(f"../../../data/top30groups/anonLoc/combined/combined{partition}.csv")
    label_index = {g: i for i, g in enumerate(sorted(df['gname'].unique()))}


    from sklearn.preprocessing import StandardScaler

    # Columns to exclude from scaling
    exclude_cols = ['gname']

    # Columns to scale
    scale_cols = [col for col in df.columns if col not in exclude_cols]

    # Scale only selected columns
    scaler = StandardScaler()
    df[scale_cols] = scaler.fit_transform(df[scale_cols])

    feature_cols = [c for c in df.columns if c != "gname"]
    non_geo_features = torch.tensor(
        df[feature_cols].astype(float).fillna(0).values,
        dtype=torch.float32
    )

    X, y_nrf, train_mask, val_mask, test_mask, index_to_label = build_nrf_data(df, label_index, feature_cols)

    # -------- Stage 1: Find best params (1500 epochs) --------
    best_score_stage1 = -1
    best_params = None

    for combo in sampled_dicts:
        args = {
            **combo,
            'partition': f"gtd{partition}",
            'n_class': len(label_index),
            'epochs': 1500,
            'final_evaluation': False,
            'verbose': False
        }

        print(f"Running config: {args}")
        acc, _, *_ = train_joint(non_geo_features, y_nrf, train_mask, val_mask, test_mask, args, index_to_label)

        if acc > best_score_stage1:
            best_score_stage1 = acc
            best_params = combo

    # -------- Stage 2: Full run (3000 epochs) --------
    final_args = {
        **best_params,
        'partition': f"gtd{partition}",
        'n_class': len(label_index),
        'epochs': 3000,
        'final_evaluation': True
    }

    print(f"[Stage 2] Final run for partition {partition} with params: {final_args}")
    acc, epoch, precision, recall, f1, y_pred_decoded, y_true_decoded, p_micro, r_micro, f1_micro, p_macro, r_macro, f1_macro, auc_w, auc_mi, auc_ma, epoch_logs = train_joint(
        non_geo_features, y_nrf, train_mask, val_mask, test_mask, final_args, index_to_label
    )

    best_scores.append(acc)
    best_paramsets.append(best_params)
    best_runs.append({
        "args": final_args,
        "acc": acc,
        "epoch": epoch,
        "y_pred": y_pred_decoded,
        "y_true": y_true_decoded,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "micro": (p_micro, r_micro, f1_micro),
        "macro": (p_macro, r_macro, f1_macro),
        "auroc": (auc_w, auc_mi, auc_ma),
        "epoch_logs": epoch_logs
    })


Running config: {'lr': 0.01, 'n_tree': 80, 'tree_depth': 12, 'tree_feature_rate': 0.3, 'feat_dropout': 0.0, 'embed_dim': 32, 'partition': 'gtd300', 'n_class': 30, 'epochs': 1500, 'final_evaluation': False, 'verbose': False}
Epoch 000 | NDF Loss: 3.4012 | Val Acc: 0.2094
Epoch 050 | NDF Loss: 2.5614 | Val Acc: 0.4533
Epoch 100 | NDF Loss: 1.9669 | Val Acc: 0.4894
Epoch 150 | NDF Loss: 1.5790 | Val Acc: 0.5044
Epoch 200 | NDF Loss: 1.3507 | Val Acc: 0.5094
Epoch 250 | NDF Loss: 1.2146 | Val Acc: 0.5139
Epoch 300 | NDF Loss: 1.1277 | Val Acc: 0.5139
Epoch 350 | NDF Loss: 1.0705 | Val Acc: 0.5156
Early stopping at epoch 373
Best validation acc: 0.5172 @ epoch 273
Running config: {'lr': 0.01, 'n_tree': 80, 'tree_depth': 10, 'tree_feature_rate': 0.5, 'feat_dropout': 0.2, 'embed_dim': 32, 'partition': 'gtd300', 'n_class': 30, 'epochs': 1500, 'final_evaluation': False, 'verbose': False}
Epoch 000 | NDF Loss: 3.4012 | Val Acc: 0.2033
Epoch 050 | NDF Loss: 2.5962 | Val Acc: 0.4533
Epoch 100 | ND

In [7]:
#gtd100
#Running config: {'lr': 0.001, 'n_tree': 40, 'tree_depth': 8, 'tree_feature_rate': 0.3, 'feat_dropout': 0.1, 'embed_dim': 32, 'partition': 'gtd100', 'n_class': 30, 'epochs': 1500, 'final_evaluation': False}
#Best validation acc: 0.4300 @ epoch 686


#gtd200
#Running config: {'lr': 0.001, 'n_tree': 80, 'tree_depth': 9, 'tree_feature_rate': 0.5, 'feat_dropout': 0.2, 'embed_dim': 64, 'partition': 'gtd300', 'n_class': 30, 'epochs': 1500, 'final_evaluation': False}
#Best validation acc: 0.5411 @ epoch 906


In [8]:
# Save best results
for i, p in enumerate(partitions):
    args = best_runs[i]["args"]
    os.makedirs(f"Results{p}", exist_ok=True)

    results_path = f"Results{p}/Results_{p}_prediction"
    with open(results_path, "w") as f:
        f.write(f"Best acc: {best_runs[i]['acc']:.4f} at epoch {best_runs[i]['epoch']} for {p} prediction\n")
        f.write(f"Config: {args}\n")
        f.write(f"Weighted Precision: {best_runs[i]['precision']:.4f}, Recall: {best_runs[i]['recall']:.4f}, F1: {best_runs[i]['f1']:.4f}\n")
        f.write(f"Macro Precision: {best_runs[i]['macro'][0]:.4f}, Recall: {best_runs[i]['macro'][1]:.4f}, F1: {best_runs[i]['macro'][2]:.4f}\n")
        f.write(f"Micro Precision: {best_runs[i]['micro'][0]:.4f}, Recall: {best_runs[i]['micro'][1]:.4f}, F1: {best_runs[i]['micro'][2]:.4f}\n")
        f.write(f"AUROC Weighted: {best_runs[i]['auroc'][0]:.4f}, Micro: {best_runs[i]['auroc'][1]:.4f}, Macro: {best_runs[i]['auroc'][2]:.4f}\n")

    log_path = f"Results{p}/epoch_logs_{p}_prediction"
    with open(log_path, "w") as f:
        f.write('\n'.join(f"{x:.4f}" for x in best_runs[i]['epoch_logs']))

    y_preds.append(best_runs[i]['y_pred'])
    y_trues.append(best_runs[i]['y_true'])

In [9]:
#test 0.9355495572090149
#Running config: {'lr': 0.001, 'n_tree': 40, 'tree_depth': 10, 'tree_feature_rate': 0.5, 'feat_dropout': 0.1, 'embed_dim': 64, 'partition': 'gtd100', 'n_class': 30, 'epochs': 3000, 'final_evaluation': True}
#Early stopping at epoch 779
#Best validation acc: 0.9331 @ epoch 679
#{'args': {'lr': 0.001, 'n_tree': 100, 'tree_depth': 10, 'tree_feature_rate': 0.5, 'feat_dropout': 0.1, 'embed_dim': 32, 'partition': 'gtd100', 'n_class': 30, 'epochs': 3000, 'final_evaluation': True}, 'acc': 0.9355495572090149,

In [10]:
print(best_run)

NameError: name 'best_run' is not defined

In [ ]:
"""
default_args = {
    'partition': f"gtd{partition}",
    'embed_dim': 16,
    'lr': 0.001,
    'epochs': 1000,
    'feat_dropout': 0,
    'n_tree': 80,
    'tree_depth': 10,
    'tree_feature_rate': 0.5,
    'n_class': len(label_index),
    'final_evaluation': True
}
0.9287652969360352

"""

In [ ]:
best_acc

In [ ]:
"""
Best acc: 0.9205 at epoch 750 for weaptype1 prediction
Weighted Precision: 0.9242, Recall: 0.9205, F1: 0.9191
Macro Precision: 0.9176, Recall: 0.9107, F1: 0.9105
Micro Precision: 0.9205, Recall: 0.9205, F1: 0.9205
AUROC Weighted: 0.9967, Micro: 0.9970, Macro: 0.9963

"""

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_pred_decoded, y_true_decoded))

In [ ]:
def plot_confusion_matrix(y_true, y_pred, labels, continuous_col):
    from sklearn.metrics import confusion_matrix
    import matplotlib.pyplot as plt
    import seaborn as sns
    import numpy as np

    cm = confusion_matrix(y_true, y_pred, labels=labels)
    cm_normalized = cm.astype('float') / cm.sum(axis=1, keepdims=True)

    plt.figure(figsize=(18, 16))
    sns.heatmap(cm_normalized,
                annot=True,
                fmt=".2f",
                xticklabels=labels,
                yticklabels=labels,
                cmap="viridis",
                square=True,
                linewidths=0.5,
                cbar_kws={"shrink": 0.8})

    plt.title(f"Normalized Confusion Matrix", fontsize=18)
    plt.xlabel("Predicted Label", fontsize=14)
    plt.ylabel("True Label", fontsize=14)
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()

    # Save the figure
    save_path = f"Results{partition}/cm_{partition}_{continuous_col}.png"
    plt.savefig(save_path, dpi=300)
    plt.close()

    print(f"Saved confusion matrix for partition {partition} to {save_path}")


In [ ]:
for i in range(len(continuous_cols)):
    plot_confusion_matrix(y_preds[i], y_trues[i], sorted(df['gname'].unique()), continuous_cols[i])